[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/36_int8_quantization_solution.ipynb)

# ✅ Solution: INT8 Quantized Linear

Implement a **post-training quantized linear layer** using INT8 weights.

### Signature
```python
class Int8Linear(nnx.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax.numpy as jnp
from flax import nnx
class Int8Linear(nnx.Module):
    def __init__(self,weight,bias=None):
        scale=jnp.max(jnp.abs(weight),axis=1,keepdims=True)/127.; self.weight_int8=nnx.Variable(jnp.clip(jnp.round(weight/(scale+1e-10)),-128,127).astype(jnp.int8)); self.scale=nnx.Variable(scale); self.bias=nnx.Param(jnp.array(bias)) if bias is not None else None
    def __call__(self,x):
        y=x@(self.weight_int8.value.astype(jnp.float32)*self.scale.value).T
        return y if self.bias is None else y+self.bias.value


In [ ]:
# Verify
print(Int8Linear)


In [ ]:
from jax_judge import check
check("int8_quantization")
